# Complete Tutorial: From Configuration to Execution

This comprehensive tutorial demonstrates the complete workflow for using the AlgoDisco framework to discover and optimize algorithms using Large Language Models (LLMs).

## Table of Contents
1. [Creating a Template Program](#1-creating-a-template-program)
2. [Creating a Task Description](#2-creating-a-task-description)
3. [Creating an Evaluator](#3-creating-an-evaluator)
4. [Creating a YAML Configuration](#4-creating-a-yaml-configuration)
5. [Running the Search](#5-running-the-search)
6. [Understanding the Output](#6-understanding-the-output)
7. [Troubleshooting Guide](#7-troubleshooting-guide)
8. [Summary](#8-summary)


## 1. Creating a Template Program

A **template program** serves as the skeletal structure of your algorithm. It defines:
- The function signature (name, parameters, return type)
- Docstring with description of what the function should do
- Placeholder comments indicating where implementation is needed

The LLM will use this template to understand what it needs to implement. A well-designed template significantly improves the quality of generated solutions.

### Key Elements of a Good Template:

- **Clear function signature**: Include type hints for better LLM understanding
- **Comprehensive docstring**: Explain the expected behavior, inputs, and outputs
- **TODO comments**: Mark specific areas where implementation is required
- **Example usage**: Include example inputs and outputs if helpful


In [ ]:
# Template program defining the algorithm skeleton
# The template includes function signature and partial implementation

template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """
    Sort a list of integers in ascending order.

    Args:
        lst: A list of integers to sort

    Returns:
        A new list with integers sorted in ascending order
    """
    # TODO: Implement the sorting algorithm
    # Your code here
    pass
'''

# Save the template program to a file
with open('template_sorting.py', 'w') as f:
    f.write(template_code)

print("Template program saved to template_sorting.py")


## 2. Creating a Task Description

The **task description** provides the LLM with detailed instructions on what problem it needs to solve. Unlike the template (which focuses on *how* to structure the code), the task description focuses on *what* the code should accomplish.

### Components of an Effective Task Description:

1. **Role definition**: Tell the LLM who it is (e.g., "You are an expert programmer")
2. **Core task**: Clearly state what needs to be implemented
3. **Requirements**: List specific constraints and requirements
4. **Examples**: Provide input/output examples for clarity
5. **Constraints**: Specify what NOT to do (e.g., don't use built-in functions)
6. **Edge cases**: Mention how to handle special cases

### Best Practices:
- Be specific about input/output formats
- List all constraints explicitly
- Include concrete examples
- Mention performance considerations if relevant


In [ ]:
# Task description tells the LLM what problem to solve
# This is used to guide the LLM in generating appropriate solutions

task_description = '''
You are an expert programmer. Your task is to implement a sorting algorithm.

Requirements:
1. The function should take a list of integers
2. Return a new list sorted in ascending order
3. Do not modify the original list
4. Handle edge cases: empty list, single element

Example:
Input: [3, 1, 4, 1, 5]
Output: [1, 1, 3, 4, 5]

Important:
- Do not use built-in sorted() or list.sort()
- Implement your own sorting algorithm (e.g., bubble sort, quick sort)
'''

# Save the task description to a file
with open('task_description.txt', 'w') as f:
    f.write(task_description)

print("Task description saved to task_description.txt")


## 3. Creating an Evaluator

The **Evaluator** is a critical component that assesses the quality of generated algorithm implementations. It:
- Executes the generated code in a sandboxed environment
- Tests the implementation against various test cases
- Computes a score based on correctness, performance, or other metrics
- Reports execution time and any errors encountered

### Designing a Good Evaluator:

1. **Test case generation**: Create diverse test cases including:
   - Normal cases (typical inputs)
   - Edge cases (empty input, single element, very large input)
   - Boundary cases (duplicate values, already sorted, reverse sorted)
2. **Timeout handling**: Set reasonable timeouts to prevent infinite loops
3. **Error reporting**: Capture and report specific error messages
4. **Scoring strategy**: Define how to compute the final score

### Evaluator Interface:

Your evaluator must implement the `evaluate_program(program_str: str) -> EvalResult` method, which returns a dictionary containing:
- `score`: A float between 0.0 and 1.0 indicating correctness
- `execution_time`: How long the program took to execute
- `error_msg`: Any error message if execution failed


In [ ]:
# Create a custom Evaluator for testing sorting algorithms
# The Evaluator executes generated code and scores based on correctness

evaluator_code = '''
import random
import subprocess
import tempfile
from algodisco.base.evaluator import Evaluator, EvalResult

class SortingEvaluator(Evaluator):
    """Evaluator for sorting algorithm correctness"""

    def __init__(self, num_tests=100):
        self.num_tests = num_tests
        self.test_cases = self._generate_test_cases()

    def _generate_test_cases(self):
        """Generate random test cases with varying sizes and values"""
        cases = []
        for _ in range(self.num_tests):
            n = random.randint(0, 20)
            case = random.sample(range(-100, 100), n)
            expected = sorted(case)
            cases.append((case, expected))
        return cases

    def evaluate_program(self, program_str: str) -> EvalResult:
        """Evaluate the generated program on test cases"""
        try:
            # Write program to a temporary file for execution
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name

            # Test the program on a subset of test cases (first 20)
            correct = 0
            for inputs, expected in self.test_cases[:20]:
                test_code = f"""
import sys
sys.path.insert(0, '.')
exec(open("{temp_path}").read())
result = sort_list({inputs})
print(result == {expected})
"""
                exec_result = subprocess.run(
                    ['python', '-c', test_code],
                    capture_output=True,
                    timeout=5
                )
                if exec_result.returncode == 0 and 'True' in exec_result.stdout.decode():
                    correct += 1

            # Calculate score as ratio of correct answers
            score = correct / min(20, len(self.test_cases))

            return {
                "score": score,
                "execution_time": 0.0,
                "error_msg": None
            }

        except subprocess.TimeoutExpired:
            return {
                "score": 0.0,
                "execution_time": 5.0,
                "error_msg": "Timeout"
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200]
            }
'''

with open('sorting_evaluator.py', 'w') as f:
    f.write(evaluator_code)

print("Evaluator saved to sorting_evaluator.py")


## 4. Creating a YAML Configuration

The **YAML configuration file** ties all components together and configures the entire search pipeline. This is the main entry point for customizing the algorithm discovery process.

### Configuration Sections

#### Method Configuration
| Parameter | Description | Recommended Value |
|-----------|-------------|-------------------|
| `template_program_path` | Path to the template program file | `"template_sorting.py"` |
| `task_description_path` | Path to the task description file | `"task_description.txt"` |
| `language` | Programming language | `"python"` |
| `num_samplers` | Number of parallel sampling processes | `2-4` |
| `num_evaluators` | Number of parallel evaluation processes | `2-4` |
| `examples_per_prompt` | Number of examples to include in each prompt | `1-3` |
| `samples_per_prompt` | Number of samples to generate per prompt | `1-5` |
| `max_samples` | Maximum total samples to generate | `100-1000` |
| `llm_max_tokens` | Maximum tokens in the LLM response | `512-2048` |
| `llm_timeout_seconds` | Timeout for LLM API calls | `60` |
| `db_num_islands` | Number of islands in the island model | `3-10` |
| `db_reset_period` | Seconds before island reset | `3600-7200` |
| `keep_metadata_keys` | Metadata keys to preserve in logs | `["sample_time", "eval_time"]` |

#### LLM Configuration
| Parameter | Description | Example |
|-----------|-------------|---------|
| `class_path` | Full path to the LLM provider class | `"algodisco.providers.llm.openai_api.OpenAIAPI"` |
| `model` | Model name | `"gpt-4o-mini"` |
| `api_key` | API key | `null` |
| `base_url` | API base URL | `"https://api.openai.com/v1"` |

#### Evaluator Configuration
| Parameter | Description | Example |
|-----------|-------------|---------|
| `class_path` | Full path to the evaluator class | `"sorting_evaluator.SortingEvaluator"` |
| `kwargs` | Constructor arguments | `{"num_tests": 100}` |

#### Logger Configuration
| Parameter | Description | Example |
|-----------|-------------|---------|
| `class_path` | Full path to the logger class | `"algodisco.providers.logger.pickle_logger.BasePickleLogger"` |
| `logdir` | Directory for log files | `"logs/sorting_search"` |

### Environment Variables

For the current AlgoDisco config loader, the usual pattern is:

- Set `OPENAI_API_KEY` in your shell
- Keep `api_key: null` in YAML
- Set `base_url` explicitly if needed


In [ ]:
# YAML configuration file that connects all components
# This defines the complete search pipeline settings

yaml_config = '''
# Method Configuration
method:
  template_program_path: "template_sorting.py"
  task_description_path: "task_description.txt"
  language: "python"
  num_samplers: 2
  num_evaluators: 2
  examples_per_prompt: 2
  samples_per_prompt: 2
  max_samples: 100
  llm_max_tokens: 1024
  llm_timeout_seconds: 60
  db_num_islands: 5
  db_reset_period: 7200
  keep_metadata_keys:
    - sample_time
    - eval_time
    - execution_time
    - error_msg

# LLM Configuration
llm:
  class_path: "algodisco.providers.llm.openai_api.OpenAIAPI"
  kwargs:
    model: "gpt-4o-mini"
    api_key: null
    base_url: "https://api.openai.com/v1"

# Evaluator Configuration
evaluator:
  class_path: "sorting_evaluator.SortingEvaluator"
  kwargs:
    num_tests: 100

# Logger Configuration
logger:
  class_path: "algodisco.providers.logger.pickle_logger.BasePickleLogger"
  kwargs:
    logdir: "logs/sorting_search"
'''

with open('config.yaml', 'w') as f:
    f.write(yaml_config)

print("YAML configuration saved to config.yaml")
print("Set OPENAI_API_KEY in your shell before running.")


## 5. Running the Search

Now that all components are configured, you can run the algorithm search in two common ways.

### Method 1: Command Line

The most direct option is the module entry point:

```bash
python -m algodisco.methods.funsearch.main_funsearch --config config.yaml
```

### Method 2: Python API

For more control, create the config and components directly in Python:

```python
import os

from algodisco.methods.funsearch.config import FunSearchConfig
from algodisco.methods.funsearch.search import FunSearch
from algodisco.providers.llm.openai_api import OpenAIAPI
from sorting_evaluator import SortingEvaluator

config = FunSearchConfig(
    template_program="...",
    task_description="...",
    language="python",
    max_samples=100,
)

llm = OpenAIAPI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
)

evaluator = SortingEvaluator(num_tests=100)
search = FunSearch(config=config, llm=llm, evaluator=evaluator)
search.run()
```

### Prerequisites for Running

Before running, ensure:
1. The template file exists at the configured path
2. The task description file exists at the configured path
3. The evaluator can be imported
4. The LLM credentials are set correctly
5. Required dependencies are installed


In [ ]:
# Run the search

# Method 1: Using the module entry point
# python -m algodisco.methods.funsearch.main_funsearch --config config.yaml

# Method 2: Using Python code (example below)

import os

# Set the OpenAI API key from the environment or hardcode it for quick testing
os.environ["OPENAI_API_KEY"] = "your-api-key"

from algodisco.methods.funsearch.config import FunSearchConfig
from algodisco.methods.funsearch.search import FunSearch
from algodisco.providers.llm.openai_api import OpenAIAPI

# NOTE: Actual execution still requires:
# 1. template_sorting.py exists
# 2. task_description.txt exists
# 3. sorting_evaluator.py is importable
# 4. OPENAI_API_KEY is valid

# Example execution (pseudo-code - uncomment to run):
# config = FunSearchConfig(
#     template_program="...",
#     task_description="...",
#     language="python",
#     max_samples=100,
# )
# search = FunSearch(
#     config=config,
#     llm=OpenAIAPI(
#         model="gpt-4o-mini",
#         api_key=os.environ["OPENAI_API_KEY"],
#         base_url="https://api.openai.com/v1",
#     ),
#     evaluator=SortingEvaluator(num_tests=100),
# )
# search.run()

print("Run command:")
print("python -m algodisco.methods.funsearch.main_funsearch --config config.yaml")


## 6. Understanding the Output

After running the search, AlgoDisco produces several types of output.

### Log Structure

`BasePickleLogger` stores batches in subdirectories under the configured `logdir`.
For example, you may see folders such as:

- `logs/sorting_search/algo/`
- `logs/sorting_search/database/`

Each `.pkl` file contains a batch of logged dictionaries.

### Loading and Analyzing Results

```python
import os
import pickle

algo_dir = "logs/sorting_search/algo"
if os.path.isdir(algo_dir):
    for filename in sorted(os.listdir(algo_dir)):
        if not filename.endswith(".pkl"):
            continue

        with open(os.path.join(algo_dir, filename), "rb") as f:
            batch = pickle.load(f)

        for item in batch:
            print(f"Score: {item.get('score')}")
            print(f"Program preview: {str(item.get('program', ''))[:100]}...")
            print("---")
```

### Interpreting Scores

| Score Range | Interpretation | Action |
|-------------|----------------|--------|
| `0.0 - 0.3` | Poor performance | Algorithm needs significant improvement |
| `0.3 - 0.6` | Moderate | Some test cases pass, review failed cases |
| `0.6 - 0.9` | Good | Most test cases pass, investigate edge cases |
| `0.9 - 1.0` | Excellent | Near-perfect or perfect implementation |

### Best Practices for Analysis

1. **Review errors first**: Check error messages to identify common failure patterns
2. **Analyze edge cases**: Some scores may be dragged down by specific edge cases
3. **Compare implementations**: Look at multiple high-scoring solutions for ideas
4. **Track progress**: Monitor how scores improve over iterations


## 7. Troubleshooting Guide

### Common Issues and Solutions

#### Issue: LLM API Errors

**Symptoms**: API key errors, timeout errors, rate limit errors

**Solutions**:
1. Verify API key is correctly set in environment: `echo $OPENAI_API_KEY`
2. Check API key has sufficient quota/credits
3. Increase `llm_timeout_seconds` if network is slow
4. Implement retry logic for rate limiting

#### Issue: Evaluator Failures

**Symptoms**: All scores are 0.0, or "Error" in error_msg

**Solutions**:
1. Check that the generated code has no syntax errors
2. Verify the function name matches the template
3. Ensure the return format matches expectations
4. Test the evaluator locally with a known good implementation

#### Issue: Template Not Being Followed

**Symptoms**: LLM generates code with wrong function signature or structure

**Solutions**:
1. Make the template more explicit with detailed docstrings
2. Add comments explaining exactly what is expected
3. Include example code showing the expected format
4. Increase `examples_per_prompt` to provide more guidance

#### Issue: Low Diversity in Solutions

**Symptoms**: All generated solutions look very similar

**Solutions**:
1. Increase `db_num_islands` for more diversity
2. Decrease `samples_per_prompt` to generate fewer per prompt
3. Add variety to task description examples
4. Consider using a different LLM model with more creativity

#### Issue: Slow Search Performance

**Symptoms**: Search takes very long time, low throughput

**Solutions**:
1. Increase `num_samplers` and `num_evaluators` for parallelism
2. Reduce `max_samples` if running in debug mode
3. Use a faster LLM model (e.g., gpt-4o-mini instead of gpt-4)
4. Reduce `llm_max_tokens` to get shorter responses

#### Issue: Import Errors for Custom Evaluator

**Symptoms**: "Module not found" or "Cannot import" errors

**Solutions**:
1. Ensure evaluator file is in current directory or PYTHONPATH
2. Use absolute import paths in class_path
3. Or add the evaluator's directory to PYTHONPATH before running

### Debugging Tips

1. **Run with verbose logging**: Check if the logger configuration supports verbose mode
2. **Test components individually**: Run template, evaluator, and LLM separately first
3. **Use small max_samples**: Start with `max_samples: 10` to quickly verify setup
4. **Check log files**: Examine logged errors and outputs for clues


## 8. Summary

The complete AlgoDisco workflow consists of the following steps:

| Step | Component | Purpose |
|------|-----------|---------|
| 1 | **Template Program** | Define the algorithm skeleton with function signature and partial implementation |
| 2 | **Task Description** | Provide detailed instructions to guide the LLM on what to implement |
| 3 | **Evaluator** | Create a custom evaluator to test and score generated solutions |
| 4 | **YAML Configuration** | Configure all components and search parameters in a single file |
| 5 | **Run Search** | Execute the search pipeline to discover and optimize algorithms |
| 6 | **Analyze Output** | Review logged results to understand performance and iterate |

### Key Takeaways:

- **Template + Task Description**: Together these tell the LLM both *how* to structure code and *what* to implement
- **Evaluator is Critical**: A well-designed evaluator is essential for guiding the search toward better solutions
- **YAML Configuration**: Provides a clean separation between code and configuration, making experiments reproducible
- **Iterate and Improve**: Use the troubleshooting guide and output analysis to refine your setup

### Next Steps:

1. Try modifying the template to solve a different problem
2. Experiment with different LLM models and configurations
3. Design more sophisticated evaluators with multiple metrics
4. Explore other search methods (openevolve, eoh, one_plus_one_eps)
5. Review the other tutorials in this series for more advanced topics
